# Day 57 — RadixAttention: sharing the KV cache between requests that start the same way

Yesterday's cache belonged to one sequence. In a real server most requests start with the
same few hundred tokens — a system prompt, a few examples, the earlier turns of the same
conversation — and recomputing those keys and values per request is the largest piece of
wasted work in serving.

The fix is a data structure: keep the cached tokens in a radix tree, look up the longest
prefix that is already there, and evict with LRU — but only from the leaves.

In [1]:
import math

import numpy as np

# ================================================================ 1. a workload
SYSTEM = tuple(range(1000, 1200))          # 200 tokens of system prompt


EXAMPLES = [tuple(range(2000 + 100 * i, 2000 + 100 * i + 100)) for i in range(4)]


def make_workload(n_convs=20, turns=4, seed=0, few_shot=2):
    """Requests that look like a chat server's day.

    Every request starts with the same system prompt; a request is then zero or more
    shared few-shot examples, the earlier turns of its own conversation, and finally
    the new user turn.  That is three different kinds of sharing in one list.
    """
    rng = np.random.default_rng(seed)
    reqs = []
    for c in range(n_convs):
        shared = SYSTEM + tuple(t for e in EXAMPLES[:few_shot] for t in e)
        history = ()
        for t in range(turns):
            user = tuple(int(x) for x in rng.integers(0, 900, int(rng.integers(20, 60))))
            reply = tuple(int(x) for x in rng.integers(0, 900, int(rng.integers(30, 90))))
            reqs.append({'conv': c, 'turn': t, 'tokens': shared + history + user})
            history = history + user + reply
    return reqs


reqs = make_workload()
print(f'{len(reqs)} requests, {len(set(r["conv"] for r in reqs))} conversations of 4 turns')
print('every request starts with the same', len(SYSTEM), 'token system prompt')
print('request 0 is', len(reqs[0]['tokens']), 'tokens;',
      'the next turn of the same conversation is', len(reqs[1]['tokens']))

80 requests, 20 conversations of 4 turns
every request starts with the same 200 token system prompt
request 0 is 454 tokens; the next turn of the same conversation is 578


## The tree

Each edge holds a run of token ids. Matching walks down from the root while the tokens
agree; when a request diverges halfway along an edge, the edge is split and both sides
keep sharing everything above the split.

In [2]:
# ================================================================ 2. the radix tree
class RadixNode:
    __slots__ = ('key', 'children', 'parent', 'last_used', 'refs')

    def __init__(self, key=(), parent=None):
        self.key = key                 # the tokens stored on the edge into this node
        self.children = {}             # first token of the edge -> child
        self.parent = parent
        self.last_used = 0
        self.refs = 0                  # requests currently decoding on this path


class RadixCache:
    """A radix tree whose edges hold runs of token ids.

    Every node owns `len(key)` slots in the KV pool.  Matching a new request means
    walking down from the root for as long as the tokens agree; everything matched is
    a prefill that does not have to happen.
    """

    def __init__(self, capacity=None):
        self.root = RadixNode()
        self.capacity = capacity       # in token slots; None means unlimited
        self.size = 0                  # slots in use
        self.clock = 0
        self.evicted = 0
        self.nodes = 1

    # ---------------------------------------------------------------- matching
    def match(self, tokens):
        """Longest prefix of `tokens` already in the tree, and the node it ends at."""
        self.clock += 1
        node, i = self.root, 0
        while i < len(tokens):
            child = node.children.get(tokens[i])
            if child is None:
                break
            k = child.key
            j = 0
            while j < len(k) and i + j < len(tokens) and k[j] == tokens[i + j]:
                j += 1
            node.last_used = self.clock
            if j < len(k):             # matched part of an edge: stop here
                i += j
                node = child           # partial match still counts as cached tokens
                node.last_used = self.clock
                return i, node, j
            i += len(k)
            node = child
            node.last_used = self.clock
        return i, node, len(node.key)

    # ---------------------------------------------------------------- inserting
    def insert(self, tokens):
        """Add `tokens` to the tree, splitting an edge if it diverges halfway."""
        matched, node, used = self.match(tokens)
        if used < len(node.key):       # the match stopped inside an edge: split it
            self._split(node, used)
            self.nodes += 1
        rest = tuple(tokens[matched:])
        if rest:
            child = RadixNode(rest, node)
            node.children[rest[0]] = child
            child.last_used = self.clock
            self.size += len(rest)
            self.nodes += 1
            node = child
        if self.capacity is not None:
            self._evict_until_fits()
        return matched, node

    def _split(self, node, at):
        """Turn one edge into two so that a divergent request can branch off."""
        parent = node.parent
        head, tail = node.key[:at], node.key[at:]
        mid = RadixNode(head, parent)
        mid.last_used = node.last_used
        parent.children[head[0]] = mid
        mid.children[tail[0]] = node
        node.key = tail
        node.parent = mid
        return mid

    # ---------------------------------------------------------------- eviction
    def _leaves(self):
        out, stack = [], [self.root]
        while stack:
            n = stack.pop()
            if n is not self.root and not n.children and n.refs == 0:
                out.append(n)
            stack.extend(n.children.values())
        return out

    def _evict_until_fits(self):
        """LRU, but only leaves: an internal node is a prefix somebody else is using.

        This is the one rule that makes a shared cache different from an ordinary LRU
        cache.  Evicting a node in the middle would orphan everything below it.
        """
        while self.size > self.capacity:
            leaves = self._leaves()
            if not leaves:
                break                  # everything left is in use or is a prefix
            victim = min(leaves, key=lambda n: n.last_used)
            del victim.parent.children[victim.key[0]]
            self.size -= len(victim.key)
            self.evicted += len(victim.key)
            self.nodes -= 1

    # ---------------------------------------------------------------- bookkeeping
    def total_nodes(self):
        return self.nodes


def trie_nodes(sequences):
    """How many nodes a plain character-by-character trie would need.

    The radix tree stores a run of tokens per edge; a trie stores one token per node,
    which is the same information in far more objects.
    """
    root = {}
    count = 1
    for seq in sequences:
        node = root
        for t in seq:
            if t not in node:
                node[t] = {}
                count += 1
            node = node[t]
    return count


# ================================================================ 3. what it saves
def run(reqs, capacity=None):
    """Serve the workload in order and count what had to be prefilled."""
    cache = RadixCache(capacity)
    total = reused = 0
    per_req = []
    for r in reqs:
        toks = r['tokens']
        matched, _ = cache.insert(toks)
        total += len(toks)
        reused += matched
        per_req.append({'len': len(toks), 'matched': matched})
    return {'total': total, 'reused': reused, 'hit': reused / total,
            'nodes': cache.total_nodes(), 'slots': cache.size,
            'evicted': cache.evicted, 'per_req': per_req}


def no_sharing(reqs):
    """What the same workload costs when each request keeps its own cache."""
    total = sum(len(r['tokens']) for r in reqs)
    peak = sum(len(r['tokens']) for r in reqs)
    return {'total': total, 'reused': 0, 'hit': 0.0, 'slots': peak}


res, plain = run(reqs), no_sharing(reqs)
print(f"prompt tokens, caching nothing : {plain['total']:,}")
print(f"actually new, with sharing     : {res['total'] - res['reused']:,}")
print(f"reused from the cache          : {res['reused']:,} ({res['hit'] * 100:.1f}%)")
print()
t = trie_nodes([r['tokens'] for r in reqs])
print(f"radix nodes {res['nodes']:,} vs trie nodes {t:,} ({t / res['nodes']:.0f}x more)")
print(f"token slots {res['slots']:,} vs {plain['slots']:,} without sharing")

prompt tokens, caching nothing : 47,466
actually new, with sharing     : 7,320
reused from the cache          : 40,146 (84.6%)

radix nodes 83 vs trie nodes 7,216 (87x more)
token slots 7,320 vs 47,466 without sharing


## The pool is finite

Eviction is LRU with one extra rule: only leaves may go. An internal node is a prefix
other entries are built on, and the shared trunk is exactly the node with the oldest
timestamp, because everybody matches *through* it rather than onto it.

In [3]:
def capacity_sweep(reqs, caps=(200, 400, 600, 800, 1200, 2000, 4000, None)):
    full = run(reqs)['slots']
    rows = []
    for cap in caps:
        c = full if cap is None else cap
        res = run(reqs, capacity=c)
        rows.append({'cap': c, 'frac': c / full, 'hit': res['hit'],
                     'evicted': res['evicted']})
    return rows, full


rows, full = capacity_sweep(reqs)
print(f'the whole tree would need {full:,} slots')
for r in rows:
    print(f"pool {r['cap']:>6,} ({r['frac'] * 100:>3.0f}%): hit {r['hit'] * 100:5.1f}%, "
          f"evicted {r['evicted']:,}")

the whole tree would need 7,320 slots
pool    200 (  3%): hit   0.0%, evicted 47,466
pool    400 (  5%): hit   0.0%, evicted 47,466
pool    600 (  8%): hit  81.0%, evicted 8,499
pool    800 ( 11%): hit  84.7%, evicted 6,482
pool  1,200 ( 16%): hit  84.7%, evicted 6,168
pool  2,000 ( 27%): hit  84.7%, evicted 5,264
pool  4,000 ( 55%): hit  84.7%, evicted 3,297
pool  7,320 (100%): hit  84.6%, evicted 0


## Two ways to share a prefix that should not be shared

In [4]:
# ================================================================ 4. the two traps
def prefix_on_text_not_tokens():
    """Matching on the text and matching on the tokens are not the same thing.

    A tokeniser merges across what looks like a boundary, so two requests whose text
    shares a prefix can have token sequences that diverge one token earlier - and a
    cache keyed on text would hand back keys and values for a token that was never
    there.
    """
    # a toy merge table: "ab" is one token, and so is "abc"
    vocab = {'a': 1, 'b': 2, 'c': 3, 'ab': 4, 'abc': 5, ' ': 6, 'd': 7}

    def encode(s):
        out, i = [], 0
        while i < len(s):
            for n in (3, 2, 1):        # longest match wins, as a real tokeniser does
                if s[i:i + n] in vocab:
                    out.append(vocab[s[i:i + n]])
                    i += n
                    break
        return tuple(out)

    a, b = 'abc', 'abd'
    ta, tb = encode(a), encode(b)
    text_prefix = 0
    while text_prefix < min(len(a), len(b)) and a[text_prefix] == b[text_prefix]:
        text_prefix += 1
    tok_prefix = 0
    while tok_prefix < min(len(ta), len(tb)) and ta[tok_prefix] == tb[tok_prefix]:
        tok_prefix += 1
    return {'a': a, 'b': b, 'ta': ta, 'tb': tb,
            'text_prefix': text_prefix, 'token_prefix': tok_prefix}


def evict_internal_node(reqs):
    """What happens if eviction is allowed to take an internal node.

    LRU on its own will happily evict the system prompt, because it was inserted first
    and nothing touches that node again directly - and everything hanging off it stops
    being reachable.
    """
    cache = RadixCache()
    for r in reqs[:8]:
        cache.insert(r['tokens'])
    before = run(reqs[:8])['hit']
    # pick the oldest node that is NOT a leaf, the way a naive LRU would
    stack, internal = [cache.root], []
    while stack:
        n = stack.pop()
        if n is not cache.root and n.children:
            internal.append(n)
        stack.extend(n.children.values())
    victim = min(internal, key=lambda n: n.last_used)
    orphaned = 0
    stack = [victim]
    while stack:
        n = stack.pop()
        orphaned += len(n.key)
        stack.extend(n.children.values())
    return {'victim_len': len(victim.key), 'orphaned': orphaned,
            'children': len(victim.children), 'hit_before': before}


tp = prefix_on_text_not_tokens()
print(f'"{tp["a"]}" -> {tp["ta"]}   "{tp["b"]}" -> {tp["tb"]}')
print(f'{tp["text_prefix"]} characters in common, {tp["token_prefix"]} tokens in common')
print()
ev = evict_internal_node(reqs)
print(f"oldest node by LRU holds {ev['victim_len']} tokens and has {ev['children']} children")
print(f"tokens orphaned if it is evicted: {ev['orphaned']}")

"abc" -> (5,)   "abd" -> (4, 7)
2 characters in common, 0 tokens in common

oldest node by LRU holds 124 tokens and has 1 children
tokens orphaned if it is evicted: 379


## The assertions

In [5]:
# ================================================================ 5. assertions
def check():
    reqs = make_workload()

    # sharing finds a lot, and finds strictly more than nothing
    res = run(reqs)
    assert 0.5 < res['hit'] < 1.0
    assert res['slots'] < sum(len(r['tokens']) for r in reqs)

    # the first request of all can match nothing
    assert res['per_req'][0]['matched'] == 0

    # the second turn of a conversation matches at least its own first turn
    second = next(i for i, r in enumerate(reqs) if r['turn'] == 1)
    assert res['per_req'][second]['matched'] >= len(SYSTEM)

    # a radix tree needs far fewer nodes than a trie over the same sequences
    t = trie_nodes([r['tokens'] for r in reqs])
    assert res['nodes'] < t / 10

    # a smaller pool means more eviction and a lower hit rate
    rows, _ = capacity_sweep(reqs)
    assert rows[0]['hit'] < rows[-1]['hit']
    assert rows[0]['evicted'] > 0 and rows[-1]['evicted'] == 0

    # matching text is not matching tokens
    tp = prefix_on_text_not_tokens()
    assert tp['text_prefix'] == 2 and tp['token_prefix'] == 0

    # an internal node has children that would be orphaned with it
    ev = evict_internal_node(reqs)
    assert ev['children'] >= 1 and ev['orphaned'] > ev['victim_len']
    return True


print('all assertions passed' if check() else 'FAILED')

all assertions passed
